# Project 01 (basic) — An n-gram language model

**Module 08 — NLP 1**

You build a statistical **language model** on real text (*The Adventures of
Sherlock Holmes*): n-gram counts, **add-k smoothing**, **interpolation**,
evaluation by **perplexity** and **text generation** by sampling (script part 1).

You experience directly: why unsmoothed models fail, why higher n-grams have a
lower perplexity (until the data become too thin) and why mixing (interpolation)
works best.

## Setup
Only the standard library. The first cell **downloads the corpus** from Project
Gutenberg (about 600 KB) into `datasets/` and caches it (the file is not checked
in, via `.gitignore`). Select the kernel of the repository `.venv` and run the
cells in order; then solve tasks 1–3.

## Part A — Data and preprocessing (given)
Load the corpus, split it into sentences and tokens, split train/test, build the
vocabulary with `<unk>`/`<s>`/`</s>` and count the n-grams.

In [1]:
# ---- Load the corpus (real text: "The Adventures of Sherlock Holmes") -----
import os, re, urllib.request, math, random
from collections import Counter, defaultdict

DATA_DIR = "datasets"
os.makedirs(DATA_DIR, exist_ok=True)
CORPUS = os.path.join(DATA_DIR, "sherlock.txt")
URL = "https://www.gutenberg.org/files/1661/1661-0.txt"

if not os.path.exists(CORPUS):
    print("Downloading the corpus from Project Gutenberg ...")
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    raw = urllib.request.urlopen(req).read().decode("utf-8", errors="ignore")
    # remove the Gutenberg boilerplate (licence etc.)
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    with open(CORPUS, "w", encoding="utf-8") as f:
        f.write(raw)
    print("saved:", CORPUS)

text = open(CORPUS, encoding="utf-8").read()
print(f"Corpus length: {len(text):,} characters")

Corpus length: 562,222 characters


In [2]:
# ---- Tokenization and sentence segmentation --------------------------------
def sentences(text):
    """A very simple segmentation: split sentences at .!?, then into words."""
    text = text.replace("\n", " ")
    for raw in re.split(r"[.!?]+", text):
        toks = re.findall(r"[a-z']+", raw.lower())
        if toks:
            yield toks

sents = list(sentences(text))
random.seed(42); random.shuffle(sents)
split = int(0.9 * len(sents))
train_sents, test_sents = sents[:split], sents[split:]
print(f"{len(sents):,} sentences  ->  {len(train_sents):,} train / {len(test_sents):,} test")
print("Example:", train_sents[0][:12])

7,290 sentences  ->  6,561 train / 729 test
Example: ['one', 'more', 'question']


In [3]:
# ---- The vocabulary with <unk>, <s>, </s> ----------------------------------
# Words that occur only ONCE in training become <unk> -> the model learns a
# distribution for unknown words, and the perplexity stays finite.
BOS, EOS, UNK = "<s>", "</s>", "<unk>"

train_counts = Counter(w for s in train_sents for w in s)
vocab = {w for w, c in train_counts.items() if c >= 2}
vocab |= {BOS, EOS, UNK}
print(f"Vocabulary size |V| = {len(vocab):,}")

def normalize(sent):
    return [BOS] + [w if w in vocab else UNK for w in sent] + [EOS]

train = [normalize(s) for s in train_sents]
test = [normalize(s) for s in test_sents]

Vocabulary size |V| = 4,127


In [4]:
# ---- N-gram counts (unigram, bigram, trigram) ------------------------------
uni = Counter()
bi = defaultdict(Counter)     # bi[w1][w2] = C(w1,w2)
tri = defaultdict(Counter)    # tri[(w1,w2)][w3] = C(w1,w2,w3)

for s in train:
    for i, w in enumerate(s):
        uni[w] += 1
        if i >= 1:
            bi[s[i-1]][w] += 1
        if i >= 2:
            tri[(s[i-2], s[i-1])][w] += 1

N_uni = sum(uni.values())
V = len(vocab)
print(f"Tokens in total N = {N_uni:,},  |V| = {V:,}")
print("most frequent words:", uni.most_common(8))

Tokens in total N = 108,670,  |V| = 4,127
most frequent words: [('<s>', 6561), ('</s>', 6561), ('the', 5068), ('<unk>', 3329), ('i', 2759), ('and', 2724), ('to', 2466), ('a', 2398)]


### Task 1 — Add-k probabilities and perplexity
Implement the smoothed probabilities (1a) and the perplexity (1b). Compare the
uni-/bi-/trigram.

In [5]:
# ---- Add-k smoothed probabilities ------------------------------------------
def p_unigram(w, k=1.0):
    return (uni[w] + k) / (N_uni + k * V)

def p_bigram(w, w1, k=1.0):
    return (bi[w1][w] + k) / (uni[w1] + k * V)

def p_trigram(w, w1, w2, k=1.0):
    return (tri[(w2, w1)][w] + k) / (sum(tri[(w2, w1)].values()) + k * V)

# small plausibility checks
print("P(holmes | mr) =", round(p_bigram("holmes", "mr"), 5))
print("P(the)         =", round(p_unigram("the"), 5))

P(holmes | mr) = 0.00023
P(the)         = 0.04494


In [6]:
# ---- Perplexity ------------------------------------------------------------
def perplexity(sentences, prob_fn):
    """PP = exp( -1/N * sum log P(w_i | context) ).  prob_fn(sent, i) returns
    P(w_i | the preceding words) for a position i>=1."""
    log_sum, N = 0.0, 0
    for s in sentences:
        for i in range(1, len(s)):
            p = prob_fn(s, i)
            log_sum += math.log(p)
            N += 1
    return math.exp(-log_sum / N)

pp_uni = perplexity(test, lambda s, i: p_unigram(s[i]))
pp_bi  = perplexity(test, lambda s, i: p_bigram(s[i], s[i-1]))
pp_tri = perplexity(test, lambda s, i: p_trigram(s[i], s[i-1], s[i-2]) if i >= 2
                                        else p_bigram(s[i], s[i-1]))
print(f"Perplexity (add-1):  unigram {pp_uni:7.1f} | bigram {pp_bi:7.1f} | trigram {pp_tri:7.1f}")
print()
print("Observation: with add-1 the TRIGRAM is WORSE than the unigram!")
print("The reason: with a large |V|, add-1 spreads far too much mass over unseen")
print("n-grams (over-smoothing). That is exactly why one needs interpolation/Kneser-Ney.")

Perplexity (add-1):  unigram   349.4 | bigram   622.1 | trigram  2051.4

Observation: with add-1 the TRIGRAM is WORSE than the unigram!
The reason: with a large |V|, add-1 spreads far too much mass over unseen
n-grams (over-smoothing). That is exactly why one needs interpolation/Kneser-Ney.


### Task 2 — Interpolation
Mix the three orders. With a small $k$ the perplexity should fall clearly below
that of the individual models.

In [7]:
# ---- Interpolation: mixing the uni-/bi-/trigram ----------------------------
def p_interp(s, i, l1=0.1, l2=0.3, l3=0.6, k=0.01):
    w, w1 = s[i], s[i-1]
    p = l1 * p_unigram(w, k) + l2 * p_bigram(w, w1, k)
    if i >= 2:
        p += l3 * p_trigram(w, w1, s[i-2], k)
    else:
        p += l3 * p_bigram(w, w1, k)     # no trigram context at the start of a sentence
    return p

pp_interp = perplexity(test, p_interp)
print(f"Perplexity (interpolation, k=0.01): {pp_interp:7.1f}")
print("-> clearly better than any individual model with add-1.")

Perplexity (interpolation, k=0.01):   193.1
-> clearly better than any individual model with add-1.


### Task 3 — Text generation
Sample sentences from the bigram model. The text sounds "Holmes-like" but is
grammatically crude — exactly the limit of n-grams.

In [8]:
# ---- Text generation by sampling -------------------------------------------
def generate(max_len=25, seed=0):
    rng = random.Random(seed)
    sent = [BOS]
    for _ in range(max_len):
        w1 = sent[-1]
        # the candidates from the bigram model, weighted by probability
        choices = list(bi[w1].items())
        if not choices:
            break
        words, weights = zip(*choices)
        nxt = rng.choices(words, weights=weights, k=1)[0]
        if nxt == EOS:
            break
        sent.append(nxt)
    return " ".join(w for w in sent[1:] if w not in (BOS, EOS))

for seed in range(5):
    print(" *", generate(seed=seed))

 * was removed loudly <unk> he closed upon the way i poured out the present and a process of agitation
 * i must sit down beside you must confess that i wrote the spot upon my stone
 * logic rather <unk>
 * my rubber
 * my friend and yet i saw all said i have referred to be a very murderous <unk> beauty during the last long effort straightened it


## Reflection (briefly, in writing)
1. A surprise: with **add-1** the trigram is *worse* than the unigram. Why?
   (Over-smoothing: with a large $|V|$, $k\cdot|V|$ in the denominator eats the
   whole mass.) And why does interpolation with a small $k$ fix that?
2. What happens to the perplexity if you use an unsmoothed model in `perplexity`
   and a test word never occurred in that context during training?
3. Why does interpolation improve on the pure trigram? (Script: falling back to
   the lower order when the data are thin.)
4. The generated text is locally plausible but globally nonsense. Which property
   of language can n-grams not capture in principle? (An outlook on module 09.)

Answer these for yourself after you have worked through the project — there are deliberately no reference answers.